# Python API reference

The [quickstart](quickstart.ipynb) introduces the workflow; this page is a
lookup reference for the Python call and report methods. The profiler
accepts an in-memory Polars DataFrame. The Rust API and JSON schema expose
the same report concepts.

## `onceover.profile`

```python
profile(
    df, budget="standard", deadline_seconds=None, *,
    sample_rows=None, max_predictors=None, max_pairs=None,
    detectors=None, max_findings=100, max_rules=20,
    hints=None, target=None, timing=False,
)
```

`budget` selects `quick`, `standard`, or `deep`. `deadline_seconds` is a
finite, nonnegative soft limit checked between passes. A pass already
running completes. Coverage identifies work skipped after the deadline.

`sample_rows` overrides the shared encoding/FD/model row limits and the
views fitting sample. `max_predictors` limits shortlisted model inputs;
`max_pairs` limits detailed numeric-pair statistics after broad screening.
These positive integer overrides affect particular searches, not every
method's independent caps. Column counts, duplicate checks, single-column
keys, and identity residual verification still use the full input.

`detectors=None` selects the defaults. A list replaces that selection;
`[]` returns column summaries and coverage without detector findings.
Required resource passes can also execute. Discover IDs with
`onceover.available_detectors()` or the [capability guide](capabilities.ipynb).

`max_findings` is a positive report-wide limit; `max_rules` limits selected
conditional rules and can be zero. Defaults are 100 findings and 20 rules.
Other per-method and per-target limits still apply.

`target` designates a numeric response for `target_shapes`. With default
selection, it enables that pass in addition to the defaults. With an
explicit list, the pass runs only when included. This argument does not
retarget every other detector or fit a deployable prediction model.

`timing=True` records `elapsed_ms` in coverage. Timings vary between runs;
the findings are unchanged. For repeatable report JSON, leave both timing
and the wall-clock deadline disabled.

## Declared hints

`hints` is a dictionary with these supported keys:

```python
hints = {
    "time": "date",                       # time-axis column name
    "order_by": ["entity", "date"],       # lexicographic row order
    "no_gaps": ["date", "sequence_id"],   # expected complete sequences
    "cadence": {"date": "monthly"},      # calendar or numeric step
    "dimensions": ["entity", "region"],  # panel grouping columns
    "measures": ["revenue"],             # values expected in panel cells
}
```

Cadences are `daily`, `business_day`, `weekly`, `monthly`, or a positive
integer day/sequence step. List-valued hints require lists, not bare
strings. Unusable column names and types are explained in expectations
coverage. Only supply hints that describe the actual table. See
[declared expectations](../methodology/expectations.ipynb) for the exact
checks and [drift](../methodology/drift.ipynb) for the use of `time` by the
model-routing detector.

## Report data

- `report.overview`: shape, headline, type/role counts, and top claims.
- `report.columns`: column cards keyed by name.
- `report.findings`: the selected, ranked statements with their evidence.
- `report.coverage`: pass status, details, and optional timings.
- `report.groups`: findings connected through shared columns; membership
  does not assert that columns are redundant.
- `report.data`: the complete JSON-shaped dictionary, including the
  schema version and selection counts.

These Python dictionaries are live objects. Use a copy if you want to
annotate them without changing the cached Python view. Native JSON and
Markdown renderings come from the original native report.

## Find and compare statements

```python
report.select(category="dependencies", type="functional", target="total")
report.select(column="region")
report.finding("a-finding-id")
report.findings_frame()
report.diff(other_report)
```

`select` combines all supplied filters and returns complete findings in
report order. `column` matches any subject column; `target` matches the
subject target where one is defined. No filters returns all findings.
`finding` returns one finding or `None`.

`findings_frame` is a Polars index of the ledger: IDs, categories, types,
columns, inputs, targets, scores, claims, and evidence counts. Retrieve the
complete finding for its detailed metrics and scope.

`diff` treats this report as the base and the argument as the new report.
It matches stable IDs, then reports added/removed findings, changed claims
or scores, and coverage-status changes. Evidence-only changes with the
same claim and score count as unchanged. This structural comparison is
not the statistical drift detector. See [serialization](serialization.ipynb)
for a worked example.

## Render and save

```python
report.to_json()                       # complete native JSON
report.to_json(max_chars=24_000)       # budgeted view, with omissions recorded
report.to_markdown()                  # human-readable report
report.to_digest(budget_chars=4_000)   # short index into the full report
report.save("profile.json")
report.save("profile.md", fmt="markdown")
```

A size-budgeted JSON view keeps high-ranked findings in full and indexes
others, then reduces column and coverage detail if needed. It raises
`ValueError` below the minimal skeleton. A digest rolls up similar
findings and states omissions. Read the full evidence before using either
as the basis for an automated action.

The versioned contract is in the [JSON reference](../appendix/report_contract.ipynb)
and the repository's [machine-readable schema](https://github.com/fcela/onceover/blob/main/docs/report-schema.json).
